In [2]:
# Extract_FAG_All_Relationship_Graves_POJ.py
# 8/24/2025


# fileInput="FAG_Relationship_Names_New"
# fileOutput="FAG_Relationship_Names_New_Graves"

# fileIn=DirMain+"Analysis\\"+fileInput+".csv"
# fileOut=DirMain+"OutputData\\ABMC\\"+fileOutput+"_"+timestamp+".csv"

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
from geopy.distance import distance
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 

def display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])


def random_sleep_new(get_counter):
    global sleep_time_pause,sleep_range,pause_count
    get_counter+=1
    if get_counter%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print('\n')
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds   Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    time.sleep(sleep_time)    
    return get_counter

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    for iTry in range(0,2):
        response = requests.request("GET", url, headers=my_header) 
        status_code=response.status_code
        url_source = response.text 
        if status_code!=200:
            print('\tstatus_code:',status_code, 'url',url)
            time.sleep(60)
            continue
        beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
        specific_tag_text = beautifulSoupText.get_text()
        capture_json=json.loads(specific_tag_text) 
        try:
            grave_count=capture_json['total']
        except:
            specific_tag_text
            print(capture_json)
            print(f" XXXX get_grave_count Failed: Try: {iTry} \n url: {url}")
            time.sleep(60) 
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    try:
        collection_df['links']=collection_df['links'].str[0].str[2]
    except:
        collection_df['links']=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

def trim_dataframe(dataframe_df,column):
    drop_list=['AME','Anglican','antioch','Apos','Asylum','Baptist','Bishop','Calvary','Campo',\
               'Canaan','Cathedral','Catholic','Cementerio','Cemit','Centenary','Chapel','Christ',\
               'Church','Cimet','Cimiter','Cintor','CME','Cmentarz','Colored','County','Deutscher',\
               'Dominican','Ebenezer','Friedhof','Gospel','Institute','Knight','Lutheran',\
               'Masonic ','Mennonite ','Methodist','Military','Missionary','Parish ','Presbyterian',\
               'prison','Protestant','public','Resurrection','Saint','St.','State','Township','Veterans']
    for word in drop_list:
        dataframe_df.drop(dataframe_df[dataframe_df[column].str.contains(word, case=False, na=False)].index, inplace=True) 
    dataframe_df.reset_index(drop=True, inplace = True)   
    return dataframe_df

def find_next_search_year(search_year):
    global first_search_year,last_search_year
    interval_temp=int((last_year-last_search_year)/2)
    if interval_temp>0:
        interval=int(2**(math.log(interval_temp, 2)//1))
    else:
        interval=0
    search_year=last_search_year+interval+1
    first_search_year=int(search_year-interval)
    last_search_year=int(search_year+interval)
    return search_year,interval,first_search_year,last_search_year

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

blank_line='                                                                                                                                                                                           '


#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract FAG Relationship Graves'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"


fileInput="FAG_Relationship_Names_Remain_X"
filePOJInput="Cemeteries_POJ"

fileOutput="FAG_Relationship_Names_Remain_Graves"
if append==1:fileInputEvents='FAG_Relationship_Names_New_X'

fileIn=DirMain+"Analysis\\"+fileInput+".csv"
filePOJIn=DirMain+"OutputData\\Cemeteries\\"+filePOJInput+".csv"
fileOut=DirMain+"OutputData\\ABMC\\"+fileOutput+"_"+timestamp+".csv"
fileOut0=DirMain+"OutputData\\ABMC\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\ABMC\\"+fileOutput+"_Interim.csv"


# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ Input Data $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
soldiers_names_df=pd.read_csv(fileIn, encoding='ISO-8859-1')
cemeteries_poj_df=pd.read_csv(filePOJIn, encoding='ISO-8859-1')
# $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$ Input Data $$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

cemeteries_poj_df_list=['cemetery_name','cemetery_id','prob_jewish']

field_list=['lastName','firstName','middleName','fullName','nickName','birthDate','birthMonth','birthYear','deathDate','deathMonth',\
            'deathYear','birthCityName','birthStateName','birthCountryName','cemeteryName','countryName','cemeteryId','links', 'intermentHasPhoto','Parents','Spouses','Siblings','Children','plot']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

url_skip = "https://www.findagrave.com/memorial/search?ajax=true&skip=" 
url_last_name="&limit=100&firstname=&middlename=&lastname="
url_birth_year ="&birthyear="
url_year_filter="&birthyearfilter="
url_suffix="&deathyear=&deathyearfilter=&locationId=country_4&includeMaidenName=&exactName=true&linkedToName=*"

sleep_range=[5,10]
sleep_time_pause=1*60
pause_count=100
get_counter=0
cemetery_count=0
total_frame_count=0

name_count=0
total_name_count=len(soldiers_names_df)
for index, row in soldiers_names_df.iterrows():
    name_count+=1
    last_name=row['last_name']
    cemetery_grave_count=0
    start_year=1861
    last_year=1925
    interval_0=32
    interval=interval_0
    search_year=start_year+interval
    
    first_search_year=int(search_year-interval)
    last_search_year=int(search_year+interval)
    while search_year<=last_year:
        frame_count=0
        grave_frame_count=0
        extracted_count=0
        grave_count=100
        grave_count_offset=0
        print(f"\n *** {process_name:<20}  • Name: *{last_name}* • IntervaL: {interval} • Next Search Year: {first_search_year}-{search_year}-{last_search_year} ••  Name Count: {name_count}/{total_name_count} ******* ")
        # print(f"    ....List Index: {list_index} • Category: {name_category} •  ABMC Name Count: {name_list_count}")
        while grave_frame_count<grave_count:# and interval!=0:
            interval_text=str(interval)
            if interval==0:interval_text=''
            # print(f"     .... Loop 1:  • Name: {last_name} --grave_frame_count<grave_count-- • grave_count: {grave_count} •  IntervaL: {interval} • Next Search Year: {first_search_year}-{search_year}-{last_search_year} ******* ")     
            url=url_skip+str(grave_frame_count)+url_last_name+last_name+url_birth_year+str(search_year)+url_year_filter+interval_text+url_suffix
            capture_json,grave_count=get_grave_count(url)
            # print(f"     ....  • Name: *{last_name}* • grave_count: {grave_count:<6}  •  IntervaL: {interval:<2} • Next Search Year: {first_search_year}-{search_year}-{last_search_year}  ******* ") #, end='\r')                 
            get_counter=random_sleep_new(get_counter) 
            if status_code!=200:print('\tstatus_code:',status_code);break 
            loop_count=0
            while grave_count>10000 and interval>0:
                # print(f"        ===> {process_name:<20}  • Name: *{last_name}* • IntervaL: {interval} • Next Search Year: {first_search_year}-{search_year}-{last_search_year} ••  grave_count: {grave_count} ******* ")
                if interval==1:                    
                    search_year-=interval
                    interval=0
                    first_search_year=int(search_year-interval)
                    last_search_year=int(search_year+interval)
                else:
                    interval=int(interval/2)
                    search_year-=interval
                    first_search_year=int(search_year-interval)
                    last_search_year=int(search_year+interval)
                interval_text=str(interval)
                if interval==0:interval_text=''
                print(f" {blank_line}", end='\r')
                loop_count+=1
                print(f"     .... {process_name:<20}   • Name: *{last_name}* • grave_count: {grave_count:<6}  •  IntervaL: {interval:<2} • Next Search Year: {first_search_year}-{search_year}-{last_search_year} ******" , end='\r')     
                url=url_skip+str(grave_frame_count)+url_last_name+last_name+url_birth_year+str(search_year)+url_year_filter+interval_text+url_suffix
                capture_json,grave_count=get_grave_count(url) 
                  
            get_counter=random_sleep_new(get_counter)
            grave_frame_df=extract_grave_data(capture_json)            
            if len(grave_frame_df)==0:
                search_year,interval,first_search_year,last_search_year=find_next_search_year(search_year)
                continue
            # # Match to POJ Cemeteries 
            # relatives_poj_cemeteries_df=pd.merge(
            #     grave_frame_df,
            #     cemeteries_poj_df,
            #     left_on=['cemeteryName'],
            #     right_on=['cemetery_name'],    
            #     how='inner')

            # grave_frame_df=relatives_poj_cemeteries_df
            # grave_frame_df['prob_jewish']=relatives_poj_cemeteries_df['prob_jewish']
                
            grave_frame_df=clean_dataframe(grave_frame_df)
            graves_df = pd.concat([graves_df,grave_frame_df]) 
            
            grave_frame_count+=100
            extracted_count+=len(grave_frame_df)            
            cemetery_grave_count+=len(grave_frame_df)
            
            time_now_obj=datetime.datetime.now()
            time_now=time_now_obj.strftime('%H:%M:%S') 
            print(f" {blank_line}", end='\r')
            print(f"    --- {process_name} • Name: *{last_name}* • Interval: {interval:<2} •• Search Year: {first_search_year}-{search_year}-{last_search_year} •• Name Count: {name_count}/{total_name_count}"+\
                  f" • grave count: {extracted_count}/{grave_frame_count}/{grave_count} • Time: {time_now}" , end='\r')
        search_year,interval,first_search_year,last_search_year=find_next_search_year(search_year)
        if len(graves_df)>=100000:
            def timestamp():
                today=str(datetime.date.today()).split('-')
                year=today[0]
                month=today[1]
                day=today[2]
                a=str(datetime.datetime.now())
                time_split=re.split(':| ',a)
                hour=time_split[1]
                timestamp=year+month+day+hour
                return timestamp
            timestamp_int=timestamp()
            fileOutIntT=DirMain+"OutputData\\ABMC\\"+fileOutput+"_"+timestamp_int+"_Interim.csv"
            graves_df.to_csv(fileOutIntT,mode="w", index=False,header=True)
            graves_df=pd.DataFrame([])           
        graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)

        print(f"     >>>> Return Loop :  • Name: *{last_name}* • grave_count: {grave_count:<6}  •  IntervaL: {interval:<2} • Next Search Year: {first_search_year}-{search_year}-{last_search_year}  ******* ")      
    print(f"***{process_name:<20} • Name: *{last_name}*  ••  Grave Count: {cemetery_grave_count} • ••  Name Count: {name_count}/{total_name_count} •• Size: {len(graves_df)}\n")
        
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   
graves_df=clean_dataframe(graves_df)  
graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


    


**Starting ***  Extract FAG Relationship Graves  *** , Time: 2025-09-19 11:40:36.533866  **


 *** Extract FAG Relationship Graves  • Name: *Hutchinson* • IntervaL: 32 • Next Search Year: 1861-1893-1925 ••  Name Count: 1/720 ******* 
    --- Extract FAG Relationship Graves • Name: *Hutchinson* • Interval: 16 •• Search Year: 1861-1877-1893 •• Name Count: 1/720 • grave count: 4900/4900/7389 • Time: 11:54:22              

			Sleeping for 60 seconds   Sleeping till: 09/19/2025 11:55:31
     >>>> Return Loop :  • Name: *Hutchinson* • grave_count: 7389    •  IntervaL: 16 • Next Search Year: 1894-1910-1926  *******  grave count: 7389/7400/7389 • Time: 12:02:06              

 *** Extract FAG Relationship Graves  • Name: *Hutchinson* • IntervaL: 16 • Next Search Year: 1894-1910-1926 ••  Name Count: 1/720 ******* 
    --- Extract FAG Relationship Graves • Name: *Hutchinson* • Interval: 16 •• Search Year: 1894-1910-1926 •• Name Count: 1/720 • grave count: 2500/2500/9384 • Time: 12:08:55        

KeyboardInterrupt: 

In [ ]:
graves_df
